In [2]:
from pathlib import Path
from safetensors import safe_open
from transformers.utils.hub import cached_file
from transformers import AutoConfig
import torch
import json
import os
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

/home/jeromeku/verl/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_ID = "Qwen/Qwen3-0.6B"
model_cache_dir = os.path.dirname(cached_file(MODEL_ID, filename="config.json"))

In [4]:
index_file = "model.safetensors.index.json"
has_index_file = os.path.exists(os.path.join(os.path.join(model_cache_dir, index_file)))


In [5]:
model_file = Path(model_cache_dir) / "model.safetensors"
with safe_open(model_file, 'pt', 'cpu') as f:
    qkvs = {k: f.get_tensor(k) for k in f.keys() if 'layers.0' in k and any(p in k for p in ['q_proj', 'k_proj', 'v_proj'])}

In [6]:
config = AutoConfig.from_pretrained(MODEL_ID)

In [7]:
qkvs.keys()

dict_keys(['model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.v_proj.weight'])

In [15]:
def get_qkv():
    q = qkvs.get("model.layers.0.self_attn.q_proj.weight")
    k = qkvs.get("model.layers.0.self_attn.k_proj.weight")
    v = qkvs.get("model.layers.0.self_attn.v_proj.weight")
    return q, k, v

In [16]:
q, k, v = get_qkv()

def make_arange(t: torch.Tensor, dtype=None):
  t = t.clone()
  t_arr = torch.arange(t.numel(), dtype=dtype or t.dtype).reshape_as(t)
  return t_arr


In [18]:
q = make_arange(q, torch.float)
k = make_arange(k, torch.float)
v = make_arange(v, torch.float)

In [19]:
def merge_qkv(q, k, v, config):
    q, k, v = q.clone(), k.clone(), v.clone()
    
    num_key_value_heads = config.num_key_value_heads
    hidden_dim = config.hidden_size
    num_attention_heads = config.num_attention_heads

    head_dim = getattr(
        config, "head_dim", hidden_dim // num_attention_heads
    )

    q_kv_shape = q.view(num_key_value_heads, -1, hidden_dim).shape
    group_dim = head_dim * num_attention_heads // num_key_value_heads # query dim if num_kv_heads q heads
    assert q_kv_shape[1] == group_dim

    q_proj_size = q.shape[0]
    kv_proj_size = k.shape[0]
    assert q_proj_size == head_dim * num_attention_heads
    assert kv_proj_size == head_dim * num_key_value_heads

    real_num_key_value_heads = q.shape[0] // group_dim
    assert real_num_key_value_heads == num_key_value_heads

    q = q.view(
        [
            num_key_value_heads,
            group_dim,
            -1,
        ]
    )
    assert q_kv_shape == q.shape

    k = k.view([num_key_value_heads, head_dim, -1])
    v = v.view([num_key_value_heads, head_dim, -1])
    out_shape = [-1, hidden_dim]

    qkv = torch.cat([q, k, v], dim=1).view(*out_shape).contiguous()
    
    return qkv

In [20]:
qkv_merged = merge_qkv(q, k, v, config)

In [23]:
head_dim = config.head_dim
num_kv_heads = config.num_key_value_heads
num_attn_heads =config.num_attention_heads
hidden_dim = config.hidden_size


In [24]:
head_ratio = num_attn_heads // num_kv_heads
group_dim = head_dim * head_ratio

In [41]:
def check_qkv(qkv):
    qkv_merged = qkv_merged.reshape(num_kv_heads, -1, hidden_dim)
    q_slice = qkv_merged[0, :group_dim]
    k_slice = qkv_merged[0, group_dim:group_dim+head_dim]
    v_slice = qkv_merged[0, group_dim+head_dim:group_dim+2*head_dim]
    q_slice.view(-1)[:10], k_slice.view(-1)[:10]

In [42]:
hidden_states = torch.ones(1, hidden_dim, dtype=torch.float)

In [43]:
qkv = merge_qkv(q, k, v, config)

In [45]:
merged_out = hidden_states @ qkv.T

In [46]:
merged_out.shape

torch.Size([1, 4096])

In [56]:
q_proj = hidden_states @ q.T
k_proj = hidden_states @ k.T

In [62]:
q_proj = q_proj.reshape(num_kv_heads, -1)
k_proj = k_proj.reshape(num_kv_heads, -1)
q_proj.shape, k_proj.shape

(torch.Size([8, 256]), torch.Size([8, 128]))

In [50]:
q_proj_size = num_attn_heads * head_dim

In [61]:
# reshape merged into num groups dim
group_size = head_dim * (head_ratio + 2)
merged_out = merged_out.reshape(-1, group_size)
assert merged_out.shape[0] == num_kv_heads

In [68]:
q_dim_per_head = head_dim * head_ratio
q_merged = merged_out[:,:q_dim_per_head]
k_merged = merged_out[:, q_dim_per_head:q_dim_per_head + head_dim]
q_merged.shape, k_merged.shape

(torch.Size([8, 256]), torch.Size([8, 128]))

In [70]:
q_proj.equal(q_merged)

True

In [71]:
k_proj.equal(k_merged)


True

In [ ]:
# /home/jeromeku/verl/thirdparty/megatron-lm/megatron/core/transformer/attention.py